In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
# from sklearn.ensemble import RandomForestRegressor

import warnings
warnings.filterwarnings('ignore')

In [ ]:
def load_and_explore_dataset(file_path):
    print("=" * 60)
    print("LOAD AND EXPLORE DATASET")
    print("=" * 60)
    
    pd.set_option('display.float_format', lambda x: f"{x:.2f}")

    df = pd.read_csv(file_path)
    print("Shape of dataset")
    print(df.shape)
    print("\nChecking for missing value:")
    print(df.isnull().sum())
    print("\nFirst five rows:")
    print(df.head())
    print("\nDescriptive statistic:")
    print(df.describe())
    print("\nDataset info:")
    print(df.info())
    print("\nSeniorCitizen distribution:")
    print(df["SeniorCitizen"].value_counts())
    print("\nSeniorCitizen percentage distribution:")
    print(df["SeniorCitizen"].value_counts(normalize=True) * 100)

    return df

In [ ]:
def identify_fatures(df):
    print("\n" + "=" * 60)
    print("IDENTIFY FEATURES")
    print("=" * 60)

    #feature to drop
    
    feature_to_drop = [
        "customerID",
        "Churn"
    ]

    #numerical columns

    numerical_columns = [
        "tenure",
        "MonthlyCharges",
        "TotalCharges"
    ]

    #categorical columns

    categories_columns = [
        "Partner",
        "gender",
        "Dependents",
        "PhoneService",
        "MultipleLines",
        "InternetService",
        "OnlineSecurity",
        "OnlineBackup",
        "DeviceProtection",
        "TechSupport",
        "StreamingTV",
        "StreamingMovies",
        "Contract",
        "PaperlessBilling",
        "PaymentMethod"
    ]

    print("Feature to drop:", feature_to_drop)
    print("\nNumerical columns:", numerical_columns)
    print("\nCategorical columns:", categories_columns)

    return feature_to_drop, numerical_columns, categories_columns

In [ ]:
def prepare_data(df,feature_to_drop):
    print("\n" + "=" * 60)
    print("PREPARE FEATURE DATASET")
    print("=" * 60)

    #drop unnecessary feature columns

    df_model = df.drop(columns=feature_to_drop)

    #specify label
    #feature label

    X = df_model.drop("SeniorCitizen", axis=1)

    #target label

    y = df_model['SeniorCitizen']

    print("\nFeature shape:", X.shape)
    print("\nTarget shape:", y.shape)
    print("\nFeature columns:", list(X.columns))

    return X, y, list(X.columns)

In [ ]:
def create_preprocessing_pipeline(numerical_columns, categories_columns):
    print("\n" + "=" * 60)
    print("CREATE PREPROCESSING PIPELINE")
    print("=" * 60)

    #numerical feature pipeline
    numerical_pipeline = Pipeline([
        ("scaler", StandardScaler())
    ])

    #categorical feature pipeline
    categories_pipeline = Pipeline([
        ("onehot", OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore"))
    ])

    #columns preprocessor
    preprocessor = ColumnTransformer([
        ("num", numerical_pipeline, numerical_columns),
        ("cat", categories_pipeline, categories_columns)
    ])
    return preprocessor

In [ ]:
def split_data(X,y, test_size=0.2, random_state=42):
    print("\n" + "=" * 60)
    print("SPLITTING DATA INTO TRAIN/TEST")
    print("=" * 60)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )

    print("\nTraining set size:", X_train.shape[0])
    print("\nTesting set size:", X_test.shape[0])
    print("\nTrain churn distribution:", y_train.value_counts())
    print("\nTest churn distribution:", y_test.value_counts())

    
    return X_train, X_test, y_train, y_test


In [ ]:
def create_model_pipeline(preprocessor):
    print("\n" + "=" * 60)
    print("CREATE MODEL PIPELINE")
    print("=" * 60)

    model_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42,
        class_weight='balanced',
        solver='lbfgs'
        ))
    ])

    print("\nModel pipeline created successfully")

    return model_pipeline
    

In [ ]:
def train_model(model_pipeline, X_train, y_train):
    print("\n" + "=" * 60)
    print("TRAINING MODEL")
    print("=" * 60)

    model_pipeline.fit(X_train, y_train)

    print("\nModel train successfully")
    
    #get name steps from preprocessing

    try:
        num_features = model_pipeline.named_steps['preprocessor'].transformers_[0][2]
        cat_features = model_pipeline.named_steps['preprocessor'].transformers_[1][2]

        #get named steps from OneHotEncoder
        onehot_encoder = model_pipeline.named_steps['preprocessor'].named_transformers_['cat'].named_steps['onehot']
        cat_feature_names = onehot_encoder.get_feature_names_out(cat_features)

        all_features = list(num_features) + list(cat_feature_names)

        #get coefficient

        coefficient = model_pipeline.named_steps["classifier"].coef_[0]
        print("\nTop 10 most importance features {by coefficient magnitude)")
        feature_importance = pd.DataFrame({
            'features': all_features,
            'coefficient': coefficient
        }).sort_values("coefficient", key=abs, ascending=False)
        print(feature_importance.head(10).to_string(index=False))

    except Exception as e:
        print(f"Cannot extract feature name {e}")

    
    return model_pipeline

In [ ]:
def evaluating_model(model_pipeline, X_train, X_test, y_train, y_test):
    print("\n" + "=" * 60)
    print("EVALUAING MODEL")
    print("=" * 60)

    y_train_pred = model_pipeline.predict(X_train)
    y_test_pred = model_pipeline.predict(X_test)

    y_train_proba = model_pipeline.predict_proba(X_train)[:, 1]
    y_test_proba = model_pipeline.predict_proba(X_test)[:, 1]

    #metrics

    y_train_acc_score = accuracy_score(y_train, y_train_pred)
    y_test_acc_score = accuracy_score(y_test, y_test_pred)

    y_train_roc_score = roc_auc_score(y_train, y_train_proba)
    y_test_roc_score = roc_auc_score(y_test, y_test_proba)

    print("\nMETRICS EVALUATON")
    print(f"\nTrain accuracy score: {float(y_train_acc_score):.4f}")
    print(f"\nTest accuracy score: {float(y_test_acc_score):.4f}")
    print(f"\nTrain roc score: {float(y_train_roc_score):.4f}")
    print(f"\nTest roc score: {float(y_test_roc_score):.4f}")


    print("\nCLASSIFICATION REPORT OF TRAIN SIZE")
    print("\n Classification report\n", classification_report(y_train, y_train_pred, target_names=["SeniorCitizen", "Not_SeniorCitizen"]))

    print("\nCLASSIFICATION REPORT OF TEST SIZE")
    print("\n Classification report\n", classification_report(y_test, y_test_pred, target_names=["SeniorCitizen", "Not_SeniorCitizen"]))


    #cross validation

    cv_score = cross_val_score(model_pipeline, X_train, y_train, cv=5, scoring="roc_auc")
    print("\nCross Validation (5-fold)")
    print(f"  ROU AUC:  {cv_score}")
    print(f"  Cross Validation mean: {cv_score.mean()}")
    print(f"  Cross Validation std:  {cv_score.std()}")

    metrics = {
    "y_train_pred": y_train_pred,
    "y_test_pred": y_test_pred,
    "y_train_acc_score": y_train_acc_score,
    "y_test_acc_score": y_test_acc_score,
    "y_train_roc_score": y_train_roc_score,
    "y_test_roc_score": y_test_roc_score,
    "y_train_proba": y_train_proba,
    "y_test_proba": y_test_proba,
    "cv_score": cv_score
    }

    return metrics

In [ ]:
def save_model_artifact(model_pipeline, feature_columns):
    print("\n" + "=" * 60)
    print("SAVING MODEL ARTIFACT")
    print("=" * 60)

    joblib.dump(model_pipeline, "telco_customer_model.pkl")
    print("👍 Telco customer model saved successfully")

    joblib.dump(feature_columns, "telco_customer_feature_columns.pkl")
    print("🏦 Telco Customer feature columns saved successfully")

    print("\n" + "=" * 60)
    print("ALL MODEL ARTIFACT SAVED")
    print("=" * 60)


In [ ]:
def predict_churn(customer_data):
    model_pipeline = joblib.load("telco_customer_model.pkl")
    feature_columns = joblib.load("telco_customer_feature_columns.pkl")

    customer_df = pd.DataFrame([customer_data])

    for feature in feature_columns:
        if feature not in customer_df.columns:
            raise ValueError(f"Missing require feature: {feature}")

        prediction = model_pipeline.predict(customer_df)[0]
        probability = model_pipeline.predict_proba(customer_df)[0]

        result = {
            'prediction': 'Not_SeniorCitizen' if prediction == 1 else 'SeniorCitizen',
            'Not_SeniorCitizen_probability': probability[1],
            'SeniorCitizen_probability': probability[0],
            'risk_level': "High" if probability[1] > 0.7 else "Medium" if probability[0] > 0.4 else "Low"
        }

        return result


In [ ]:
def test_predict():
    customer_data = {
        "gender": "Female",
        "Partner": "Yes",
        "tenure": 1,
        "MonthlyCharges": 29.85,
        "TotalCharges": 29.85,
        "Dependents": "No",
        "PhoneService": "No",
        "MultipleLines": "No phone service",
        "InternetService": "DSL",
        "OnlineSecurity": "No",
        "OnlineBackup": "Yes",
        "DeviceProtection": "No",
        "TechSupport": "No",
        "StreamingTV": "No",
        "StreamingMovies": "No",
        "Contract": "Month-to-month",
        "PaperlessBilling": "Yes",
        "PaymentMethod": "Electronic check"
    }

    result = predict_churn(customer_data)
    print("Prediction:", result['prediction'])
    print("Not_SeniorCitizen Probability:", result["Not_SeniorCitizen_probability"])
    print("SeniorCitizen Probability:", result["SeniorCitizen_probability"])
    print("Risk Level:", result['risk_level'])

In [ ]:
def main():
    file_path = "data/cleaned telcon-customer-churn-dataset.csv"
    
    df = load_and_explore_dataset(file_path)

    feature_to_drop, numerical_columns, categories_columns = identify_fatures(df)

    X, y, feature_columns = prepare_data(df,feature_to_drop)

    preprocessor = create_preprocessing_pipeline(numerical_columns, categories_columns)

    X_train, X_test, y_train, y_test = split_data(X, y)

    model_pipeline = create_model_pipeline(preprocessor)

    model_pipeline = train_model(model_pipeline, X_train, y_train)

    metrics = evaluating_model(model_pipeline, X_train, X_test, y_train, y_test)

    save_model_artifact(model_pipeline, feature_columns)

    test_predict()

In [ ]:
if __name__ == "__main__":
    main()